# Genera dataset y entrena

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

In [ ]:
os.makedirs('data/train', exist_ok=True)
os.makedirs('data/test', exist_ok=True)
os.makedirs('data/validation', exist_ok=True)
os.makedirs('wrong_data', exist_ok=True)

np.random.seed(42)
X = np.random.rand(1000, 4)
y = np.zeros(1000, dtype=int)

for i in range(1000):
    peak_g, jerk, audio, gyro = X[i]
    if peak_g > 0.75 and jerk > 0.7:
        y[i] = 2
    elif peak_g > 0.6 or (peak_g > 0.5 and audio > 0.6):
        y[i] = 1
    else:
        y[i] = 0

columnas = ['Peak_G', 'Jerk', 'Firma_Acustica', 'Cambio_Angular', 'Label']
df_completo = pd.DataFrame(np.column_stack((X, y)), columns=columnas)
df_completo['Label'] = df_completo['Label'].astype(int)

df_train_val, df_test = train_test_split(df_completo, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_train_val, test_size=0.25, random_state=42)

df_wrong = df_test.copy()
df_wrong['Label'] = np.random.choice([0, 1, 2], size=len(df_wrong))

df_train.to_csv('data/train/train.csv', index=False)
df_val.to_csv('data/validation/validation.csv', index=False)
df_test.to_csv('data/test/test.csv', index=False)
df_wrong.to_csv('wrong_data/wrong_data.csv', index=False)

In [ ]:
model = keras.Sequential([
    keras.layers.Dense(24, activation='relu', input_shape=(4,)),
    keras.layers.Dense(12, activation='relu'),
    keras.layers.Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
X_train_np = df_train.drop(columns=['Label']).values
y_train_np = df_train['Label'].values
X_val_np = df_val.drop(columns=['Label']).values
y_val_np = df_val['Label'].values

model.fit(X_train_np, y_train_np, epochs=40, batch_size=16, validation_data=(X_val_np, y_val_np))

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('safedrive_cu03_model.tflite', 'wb') as f:
    f.write(tflite_model)

# Lee y entrena

In [ ]:
import os
import zipfile
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

In [ ]:
with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset_extraido')

In [ ]:
archivos_csv = glob.glob('dataset_extraido/dataset/**/*.csv', recursive=True)
lista_dfs = [pd.read_csv(archivo) for archivo in archivos_csv]
df_completo = pd.concat(lista_dfs, ignore_index=True)

X = df_completo.iloc[:, :-1].values
y = df_completo.iloc[:, -1].values

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

In [ ]:
model = keras.Sequential([
    keras.layers.Dense(24, activation='relu', input_shape=(X_train.shape[1],)),
    keras.layers.Dense(12, activation='relu'),
    keras.layers.Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(X_train, y_train, epochs=40, batch_size=16, validation_data=(X_val, y_val))

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Accuracy: {accuracy}")

clases = ["Falso Positivo", "Posible Susto", "Choque Grave"]
predicciones = model.predict(X_test)

for i in range(min(10, len(y_test))):
    idx = np.argmax(predicciones[i])
    print(f"Pred: {clases[idx]} | Real: {clases[int(y_test[i])]}")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('safedrive_cu03_model.tflite', 'wb') as f:
    f.write(tflite_model)